# 🚀 AIC 2026 - Master Pipeline

Notebook này chứa toàn bộ luồng chạy hoàn chỉnh của hệ thống tìm kiếm video (Ensemble Zero-Shot + Projection Head).

### Bước 0: Kiểm tra hệ thống GPU

In [5]:
!python 0_check_system.py

=== KIỂM TRA HỆ THỐNG ===
PyTorch Version: 2.4.1+cpu
CUDA Available: False
⚠️ DANG CHAY TREN CPU - Chua nhan GPU!


### Bước 0.5: Khởi tạo Dữ liệu BTC (Tạo metadata)
Đọc thư mục chứa ảnh `data/keyframes/` để tạo ra danh sách và index tất cả các ảnh (`metadata.jsonl`). Nếu bạn đã làm rồi thì có thể bỏ qua, nếu muốn format lại thì thêm cờ `--force`.

In [8]:
!python scripts/import_btc_data.py

2026-08-15 18:04:36,133 - INFO - Scanned keyframes for 29 videos, total 22248 frames.
2026-08-15 18:04:36,429 - INFO - Loaded media-info for 873 videos.
2026-08-15 18:04:37,223 - INFO - Loaded keyframe maps for 29 videos.
2026-08-15 18:04:38,036 - INFO - [L21_V001] Processed 743 keyframes (743 organizer-mapped, 0 exact-pts, 0 fallback)
2026-08-15 18:04:39,282 - INFO - [L21_V002] Processed 870 keyframes (870 organizer-mapped, 0 exact-pts, 0 fallback)
2026-08-15 18:04:40,878 - INFO - [L21_V003] Processed 847 keyframes (847 organizer-mapped, 0 exact-pts, 0 fallback)
2026-08-15 18:04:41,930 - INFO - [L21_V005] Processed 499 keyframes (499 organizer-mapped, 0 exact-pts, 0 fallback)
2026-08-15 18:04:42,920 - INFO - [L21_V006] Processed 790 keyframes (790 organizer-mapped, 0 exact-pts, 0 fallback)
2026-08-15 18:04:43,741 - INFO - [L21_V007] Processed 624 keyframes (624 organizer-mapped, 0 exact-pts, 0 fallback)
2026-08-15 18:04:44,739 - INFO - [L21_V008] Processed 814 keyframes (814 organizer

### Bước 1: Trích xuất Đặc trưng (Ensemble 3 Mô hình)
Chạy qua toàn bộ dữ liệu ảnh đã lấy từ bước 0.5 và lưu dưới dạng `.npy`, đồng thời tạo FAISS index tạm thời (2304 chiều).

In [4]:
!python 1_extract_and_build_index.py

^C


### Bước 2: Huấn luyện Mạng thần kinh (Projection Head)
Dùng các file đặc trưng đã trích xuất ở Bước 1 kết hợp với file `captions_dummy.json` (hoặc nhãn từ BTC) để dạy cho máy học cách khớp câu tiếng Việt vào hình ảnh.

Bạn có thể đổi số `--epochs 50` thành số vòng lặp mà bạn muốn.

In [ ]:
!python 2_train_projection_head.py --epochs 50

### Bước 3: Rebuild FAISS Index
Sau khi mô hình Projection Head `projection_head_latest.pth` được lưu ở Bước 2. Ta sẽ chạy toàn bộ vector ảnh (2304d) qua mô hình này để ép về chuẩn 768d của văn bản, rồi lưu lại thành FAISS Index mới.

In [ ]:
!python 3_rebuild_projected_index.py

### Bước 4: Khởi động Backend API / Hoặc Query Trực Tiếp
Sau khi có FAISS Index hoàn chỉnh, bạn có thể chạy API để Web App Frontend kết nối tới.

In [ ]:
!python main.py

### (Tùy chọn) Chạy Test Truy vấn Trực tiếp trên Notebook

In [ ]:
import sys
import os
from backend.embedding.search_engine import VectorSearchEngine

engine = VectorSearchEngine()
engine.load_index()

query = "một bức ảnh về chiếc xe màu đỏ"
results = engine.search_single(query, top_k=5)

print(f"Kết quả cho truy vấn: {results['query_vi']}")
for r in results['results']:
    print(f"- Video ID: {r['video_id']} | Frame: {r['frame_id']} | Score: {r['score']:.4f}")